### XGBoost

In [1]:
import pandas as pd
from utils import *
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV

In [2]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24'] / 24)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V27,V28,Amount,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.133558,-0.021053,149.62,0,5.014760,0,1,0,0.0,1.0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.008983,0.014724,2.69,0,1.305626,0,1,0,0.0,1.0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.055353,-0.059752,378.66,0,5.939276,0,1,0,0.0,1.0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.062723,0.061458,123.50,0,4.824306,0,1,0,0.0,1.0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.219422,0.215153,69.99,0,4.262539,0,1,0,0.0,1.0


In [3]:
features = df.drop(['Time','Class','Amount','Hour_from_start_mod24'], axis=1).columns.tolist()
target = "Class"

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train: (181584, 33) y_train: (181584,)
X_val: (45396, 33) y_val: (45396,)
X_test: (56746, 33) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [5]:
pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
xg_model = XGBClassifier(
    n_estimators=600,
    max_depth=11,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda = 1.0,
    gamma=1.0,
    min_child_weight=7,
    tree_method = "hist",
    scale_pos_weight=neg/max(pos, 1),
    eval_metric='aucpr',
    random_state=SEED
)

xg_model.fit(X_train, y_train)

p_val_xg  = xg_model.predict_proba(X_val)[:, 1]
p_test_xg = xg_model.predict_proba(X_test)[:,1]

rs_xg_val = log_eval(y_val, p_val_xg)
rs_xg_test = log_eval(y_test, p_test_xg)
print("Validation: ")
print(rs_xg_val)
print("Test:")
print(rs_xg_test)

Validation: 
{'threshold': 0.178, 'Cost': 2495.0, 'ROC_AUC': 0.9840200213072037, 'PR_AUC': 0.7763262286733491, 'debiased_ece': 0.00032838646106565534, 'adaptive_ece': 0.0003157442098540752, 'Brier': 0.00047903129598125815}
Test:
{'threshold': 0.008, 'Cost': 2855.0, 'ROC_AUC': 0.9828558266058266, 'PR_AUC': 0.8027898839029383, 'debiased_ece': 0.0002585495722611689, 'adaptive_ece': 0.00022114501126261786, 'Brier': 0.0004461052012629807}


In [14]:
xg_sweep = sweep_thresholds(y_test, p_test_xg)
print(xg_sweep.iloc[50:101])

     threshold  precision    recall        f1  tp     tn  fp  fn    cost
50       0.050   0.557692  0.783784  0.651685  58  56626  46  16  3430.0
51       0.051   0.563107  0.783784  0.655367  58  56627  45  16  3425.0
52       0.052   0.563107  0.783784  0.655367  58  56627  45  16  3425.0
53       0.053   0.574257  0.783784  0.662857  58  56629  43  16  3415.0
54       0.054   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
55       0.055   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
56       0.056   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
57       0.057   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
58       0.058   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
59       0.059   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
60       0.060   0.580000  0.783784  0.666667  58  56630  42  16  3410.0
61       0.061   0.585859  0.783784  0.670520  58  56631  41  16  3405.0
62       0.062   0.585859  0.783784  0.670520  58  

In [20]:
result_cal = evaluate(y_test, p_test_xg)
print(f"TP={result_cal['tp']}, FP={result_cal['fp']}, FN={result_cal['fn']}, TN={result_cal['tn']}")
print(pd.Series(y_test).value_counts())

TP=56, FP=8, FN=18, TN=56664
Class
0    56672
1       74
Name: count, dtype: int64


### Caliration

In [7]:
tscv = TimeSeriesSplit(n_splits=10)

In [8]:
cal_xg = CalibratedClassifierCV(
    estimator=xg_model,
    method='isotonic',
    cv=tscv
)

cal_xg.fit(X_val, y_val)

p_test_cal_xg = cal_xg.predict_proba(X_test)[:, 1]

rs_cal_xg_test = log_eval(y_test, p_test_cal_xg)
print("Test:")
print(rs_cal_xg_test)

Test:
{'threshold': 0.015, 'Cost': 3045.0, 'ROC_AUC': 0.9578531321058495, 'PR_AUC': 0.7696164915513691, 'debiased_ece': 0.00030990995446801496, 'adaptive_ece': 0.000173298778937973, 'Brier': 0.0004490805813370974}
